### nanoGPT

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import random
%matplotlib inline

In [2]:
with open('src/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print(len(text))

1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
stoi = { ch:i for i, ch in enumerate(chars)}
itos = { i:ch for i, ch in enumerate(chars)}
encode = lambda x: [ stoi[ch] for ch in x ]
decode = lambda x: ''.join([ itos[ch] for ch in x ])
print(stoi)
print(itos)
print(encode("hii there"))
print(decode(encode("hii there")))

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i',

In [7]:
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [8]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target  = y[t]
    print(f"{t + 1} when input is {context} and target is {target}")

1 when input is tensor([18]) and target is 47
2 when input is tensor([18, 47]) and target is 56
3 when input is tensor([18, 47, 56]) and target is 57
4 when input is tensor([18, 47, 56, 57]) and target is 58
5 when input is tensor([18, 47, 56, 57, 58]) and target is 1
6 when input is tensor([18, 47, 56, 57, 58,  1]) and target is 15
7 when input is tensor([18, 47, 56, 57, 58,  1, 15]) and target is 47
8 when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) and target is 58


In [11]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix   = torch.randint(len(data) - block_size, (batch_size, ))
    x    = torch.stack([data[i   : i+block_size]   for i in ix])
    y    = torch.stack([data[i+1 : i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print("inputs:", xb.shape, '\n', xb)
print("target:", yb.shape, '\n', yb)

print("------")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t + 1]
        target  = yb[b, t]
        print(f"{t + 1} when input is {context} and target is {target}")

inputs: torch.Size([4, 8]) 
 tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
target: torch.Size([4, 8]) 
 tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
------
1 when input is tensor([24]) and target is 43
2 when input is tensor([24, 43]) and target is 58
3 when input is tensor([24, 43, 58]) and target is 5
4 when input is tensor([24, 43, 58,  5]) and target is 57
5 when input is tensor([24, 43, 58,  5, 57]) and target is 1
6 when input is tensor([24, 43, 58,  5, 57,  1]) and target is 46
7 when input is tensor([24, 43, 58,  5, 57,  1, 46]) and target is 43
8 when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) and target is 39
1 when input is tensor([44]) and target is 53
2 when input is tensor([44, 53]) and target is 56
3 when input is tensor(

In [12]:
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets = None):

        logits  = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(-1, logits.size(-1))
            targets = targets.view(-1)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # -1 means the last dimension.
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples = 1)
            idx = torch.cat((idx, idx_next), dim = 1)

        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(f"loss: {loss.item(): .4f}, \nexpected: {torch.log(torch.tensor(65)).item(): .4f}, \ndiff: {(loss.item() - torch.log(torch.tensor(65)).item()): .4f}")

print("------")

idx = torch.zeros((1, 1), dtype = torch.long)

print(decode(m.generate(idx, max_new_tokens = 100)[0].tolist()))

torch.Size([32, 65])
loss:  4.8786, 
expected:  4.1744, 
diff:  0.7042
------

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [13]:
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [14]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()

    optimizer.step()

print(loss.item())

2.382369041442871


In [19]:
idx = torch.zeros((1, 1), dtype = torch.long)

print(decode(m.generate(idx, max_new_tokens = 1000)[0].tolist()))


BR: ct
Ywit harfoul'st, ar izlor t ct.
Fo, sther:
I d tre th,-she.
Wowltothedl:
NNONANRI, aft,
STo way!
TIV:
WDUKI HANENEThe d ndean-bros g qpl mout fok yolaime do myoulato,
Mok h$ay t nch sle bionhoured whaneables mye.
For f beng tho; ar!TCald? min, wherur thaing tyoucora we d s?

Tord, g I:
Whireat pr f Yhzze?
Ther hurer cr il f aloulatspribr,
AG o otr thall oull
NEN theat.
We

GAseruisfeveg, t wild he borong s bl?
DW:
Anto' h bes whoommeulye Werngssamyou
She
BESCONG RDoleshet heked
GUE t at, sozere my par 's h, de,
I hathago pe, wowhe w?

I we be lirewin ad:
Myorodeter t ceom,

I'sustier IORCENGJULI dow

Thetindonse thel becepin.
sthtour, G hance annof k tigr tin mame
Wint pHandenomety rtonossoite pll t dsuneroncu ren hinder's,
sapp, her waige het, andanseve otoe, ngr are? tanol hat'seo hinee tiredsthorithly t ishe.
FRonis l omyo acem y r, I matolist, at then desors s crs!

RWhathin ceithin$EDI inresvenou g Burory alow lelld,
ARI't

Why heas hast t myonat,


LINTive.

Thart,
LAUM:


In [20]:
B, T, C = 4, 8, 32
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim = True)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [22]:
torch.manual_seed(1337)
tril = torch.tril(torch.ones(T, T))
wei  = torch.zeros((T, T))
wei  = wei.masked_fill(tril == 0, float('-inf'))
wei  = F.softmax(wei, dim = -1)
x = torch.randn(B, T, C)
wei @ x

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [24]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key       = nn.Linear(C, head_size, bias = False)
query     = nn.Linear(C, head_size, bias = False)
value     = nn.Linear(C, head_size, bias = False)
k         = key(x)
q         = query(x)
wei       = q @ k.transpose(-2, -1)

tril      = torch.tril(torch.ones(T, T))
# wei       = torch.zeros((T, T))
wei       = wei.masked_fill(tril == 0, float('-inf'))
wei       = F.softmax(wei, dim = -1)
x         = torch.randn(B, T, C)
# out       = wei @ x
v         = value(x)
out       = wei @ v
out.shape

torch.Size([4, 8, 16])